# LittleBit-2 QAT for Qwen3-0.6B

Enable Internet and choose the **GPU T4 x2** accelerator before training. Both GPUs train. Edit the settings below, then use **Save Version > Save & Run All**. See `train/README.md` for resume, CPU evaluation and upload.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
TINY = os.getenv("LITTLEBIT_TINY") == "1"
RUN_TRAIN = True
RUN_GGUF = False
UPLOAD = False

TRAINING_COMMIT = ""
RESUME = ""  # a path to resume.pt, or "attached" to find the single attached one
TOKEN_CACHE = ""  # optional path to tokens.pt; defaults to the file beside RESUME
RESULTS = ""  # use an attached littlebit-train directory for CPU evaluation or upload
HF_REPO = ""  # owner/repository for the optional private upload

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT = Path(os.getenv("LITTLEBIT_OUTPUT", str(WORK / "littlebit-train"))).resolve()
LEGACY_RECIPE = False
TOKENS = 64 if TINY else None  # default target: 100 million cumulative tokens
SEQ_LEN = 16 if TINY else 2048
GRAD_ACCUM = None  # sequences per update, split over both GPUs; default: 2
C4_TOKENS = None  # default: complete shard; set an integer to limit the prefix
MINUTES = 630
LOGIT_CHUNK = 128


def find_resume(setting: str, inputs: Path = Path("/kaggle/input")) -> str:
    if setting != "attached":
        if setting and not Path(setting).is_file():
            raise FileNotFoundError(f"RESUME does not exist: {setting}")
        return setting
    found = sorted(inputs.glob("**/littlebit-train/resume.pt"))
    if len(found) != 1:
        raise FileNotFoundError(
            f"expected one attached littlebit-train/resume.pt, found {len(found)}: "
            f"{[str(path) for path in found]}"
        )
    return str(found[0])


RESUME = find_resume(RESUME)
if RUN_TRAIN and (GRAD_ACCUM or (1 if LEGACY_RECIPE else 2)) % 2:
    raise ValueError("Both GPUs train, so set GRAD_ACCUM to an even number.")
if RESUME and RESULTS:
    raise ValueError("Set RESUME to train or RESULTS to evaluate, not both.")
print("Resume file:", RESUME or "none, starting a new run")

In [ ]:
LOCAL_REPO = os.getenv("LITTLEBIT_REPO")
SCRATCH = Path("/kaggle/temp")
command_env = os.environ.copy()
# The kernel's injected modules belong to its Python, not the isolated environment.
command_env.pop("PYTHONPATH", None)
command_env.pop("PYTHONHOME", None)
command_env.pop("CUDA_VISIBLE_DEVICES", None)
command_env["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"


def require_two_gpus() -> None:
    try:
        listing = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        listing = ""
    gpus = [line for line in listing.splitlines() if line.startswith("GPU ")]
    if len(gpus) != 2:
        raise RuntimeError(
            f"Training needs exactly two GPUs and this session has {len(gpus)}. "
            'Select "GPU T4 x2" under Settings > Accelerator and start again.'
        )
    for gpu in gpus:
        print(gpu, flush=True)


def run_command(
    command: list[str | Path], cwd: Path | None = None, input_text: str | None = None
) -> None:
    if input_text is not None:
        completed = subprocess.run(
            command,
            cwd=cwd,
            env=command_env,
            input=input_text,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        print(completed.stdout, end="", flush=True)
        completed.check_returncode()
        return
    with subprocess.Popen(
        command,
        cwd=cwd,
        env=command_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)


if LOCAL_REPO:
    repo = Path(LOCAL_REPO).resolve()
    python = sys.executable
else:
    if RUN_TRAIN and not TINY:
        require_two_gpus()
    SCRATCH.mkdir(parents=True, exist_ok=True)
    command_env["TMPDIR"] = str(SCRATCH)
    repo = SCRATCH / "littlebit-gguf"
    venv = SCRATCH / "littlebit-env"
    python = str(venv / "bin/python")
    upstream_repo = "https://github.com/GeniusDevam/littlebit-gguf.git"

    # Evaluation must use the commit that trained. A resumed run uses the newest code.
    revision = TRAINING_COMMIT
    summary = Path(RESULTS).resolve() / "summary.json"
    if not revision and RESULTS and summary.is_file():
        revision = json.loads(summary.read_text())["training_commit"]
    revision = revision or "train-r3"
    if not (repo / ".git").is_dir():
        run_command(["git", "init", "-q", repo])
    run_command(["git", "fetch", "--depth", "1", upstream_repo, revision], cwd=repo)
    run_command(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)

    if not Path(python).is_file():
        run_command([sys.executable, "-m", "venv", "--without-pip", venv])
    pip_install = [
        sys.executable,
        "-m",
        "pip",
        "--python",
        python,
        "install",
        "--progress-bar",
        "off",
    ]
    wheels = "cpu" if TINY or not RUN_TRAIN else "cu126"
    run_command(
        pip_install
        + [
            f"torch==2.14.0+{wheels}",
            "--index-url",
            f"https://download.pytorch.org/whl/{wheels}",
        ]
    )
    run_command(pip_install + ["-r", repo / "train/requirements.txt"])

print(
    "Source commit",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip(),
)

In [ ]:
if RUN_TRAIN:
    command = [
        python,
        "-u",
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--local-addr",
        "127.0.0.1",
        "--nproc-per-node",
        "2",
        "-m",
        "train.train_qat",
        "--output",
        OUTPUT,
        "--session-start",
        str(SESSION_START),
        "--minutes",
        str(MINUTES),
        "--seq-len",
        str(SEQ_LEN),
        "--logit-chunk",
        str(LOGIT_CHUNK),
    ]
    for flag, value in (
        ("--tokens", TOKENS),
        ("--c4-tokens", C4_TOKENS),
        ("--grad-accum", GRAD_ACCUM),
    ):
        if value is not None:
            command += [flag, str(value)]
    if LEGACY_RECIPE:
        command += ["--legacy-recipe"]
    if TOKEN_CACHE:
        command += ["--token-cache", TOKEN_CACHE]
    if RESUME:
        command += ["--resume", RESUME]
    if TINY:
        command += ["--tiny", "--cpu", "--log-every", "1"]
    if os.getenv("LITTLEBIT_UPSTREAM"):
        command += ["--upstream", os.environ["LITTLEBIT_UPSTREAM"]]
    run_command(command, cwd=repo)
    assert (OUTPUT / "littlebit/littlebit_config.json").is_file()
    assert (OUTPUT / "resume.pt").is_file()
    assert (OUTPUT / "tokens.pt").is_file()
else:
    print("Training is off.")

## Optional CPU evaluation

For a separate CPU session, attach the training output, set `RESULTS` to its path, set `RUN_TRAIN = False`, and enable `RUN_GGUF`. The cell builds the patched tools, converts both embedding types and runs perplexity on the WikiText-2 test text and the C4 validation text for both files and the BF16 teacher. It also records the CPU and a short generation sample.

In [ ]:
if RUN_GGUF:
    results = Path(RESULTS).resolve() if RESULTS else OUTPUT
    run_command(
        [
            python,
            "-u",
            "-m",
            "train.evaluate_gguf",
            results,
            WORK / "littlebit-evaluation",
        ],
        cwd=repo,
    )
else:
    print("GGUF conversion and evaluation are off.")

## Optional private upload

Set `HF_REPO`, add `HF_TOKEN` in Kaggle Secrets and grant this notebook access to it. Enable `UPLOAD` to publish the format-exact checkpoint folder to your Hugging Face repository.

In [ ]:
if UPLOAD:
    if not HF_REPO:
        raise ValueError("Set HF_REPO to owner/repository before enabling upload.")
    from kaggle_secrets import UserSecretsClient

    results = Path(RESULTS).resolve() if RESULTS else OUTPUT
    upload = """
import sys
from huggingface_hub import HfApi

try:
    api = HfApi(token=sys.stdin.read())
    api.create_repo(sys.argv[1], repo_type="model", private=True, exist_ok=True)
    if not api.model_info(sys.argv[1]).private:
        sys.exit("The destination repository must be private.")
    api.upload_folder(repo_id=sys.argv[1], folder_path=sys.argv[2], repo_type="model")
except Exception:
    sys.exit("Upload failed. Check the secret and repository permissions.")
"""
    run_command(
        [python, "-c", upload, HF_REPO, results / "littlebit"],
        input_text=UserSecretsClient().get_secret("HF_TOKEN"),
    )
else:
    print("Upload is off.")